# Laboratorio 4 — Análisis de Datos GeoEspaciales
## Monitoreo de cianobacteria en Lago Atitlán y Lago Amatitlán

## 0. Setup e instalación de librerías

Necesitamos una cuenta en **Copernicus Data Space Ecosystem** (https://dataspace.copernicus.eu/) para conectarnos con openeo.

In [27]:
# Si hace falta, descomenta para instalar
!pip install openeo geopandas rasterio xarray matplotlib folium pandas numpy scipy


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\eagi5\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [28]:
import openeo
import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt
import folium
from pathlib import Path

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)


## Datos base del laboratorio (dados por el PDF)

Acá están las coordenadas (bounding box) y fechas de cada lago. Todos usamos las mismas fechas, así están estandarizadas.

In [29]:
lagos = {
    "atitlan": {
        "west": -91.326256,
        "east": -91.07151,
        "south": 14.5948,
        "north": 14.750979,
        "fechas": [
            "2025-01-18", "2025-04-13", "2025-05-13", "2025-07-17",
            "2025-11-21", "2025-12-29", "2026-02-12", "2026-03-24",
            "2026-04-13", "2026-04-28", "2026-07-22",
        ],
    },
    "amatitlan": {
        "west": -90.638065,
        "east": -90.512924,
        "south": 14.412347,
        "north": 14.493799,
        "fechas": [
            "2025-01-28", "2025-04-15", "2025-04-28", "2025-11-24",
            "2026-01-08", "2026-02-02", "2026-02-07", "2026-03-29",
            "2026-04-13", "2026-04-28", "2026-06-19",
        ],
    },
}

for nombre, info in lagos.items():
    print(nombre, "->", len(info["fechas"]), "fechas")


atitlan -> 11 fechas
amatitlan -> 11 fechas


---
## Ejercicio 1 — Conexión con la API de Sentinel-2 (openEO)

Nos conectamos al backend de openEO del Copernicus Data Space Ecosystem. Cuando corremos esto nos pide autenticación con OIDC, abre una ventana del navegador para que nos loguemos con nuestra cuenta de Copernicus.

In [30]:
connection = openeo.connect("openeo.dataspace.copernicus.eu")
connection.authenticate_oidc() 

print("Conectado:", connection.capabilities().api_version())


Authenticated using refresh token.
Conectado: 1.2.0


---
## Ejercicio 2 — Descarga de las bandas necesarias por lago

Descargamos todas las bandas que usamos para los tres índices:
- **NDVI:** bandas B04, B08
- **NDWI:** bandas B03, B08
- **Cianobacteria (CyanoLakes Chlorophyll-a):** bandas B02, B03, B04, B05, B07, B08, B8A, B11, B12

**Unión total:** B02, B03, B04, B05, B07, B08, B8A, B11, B12 (9 bandas). Descargamos una sola vez por fecha/lago para evitar descargas duplicadas.

Hicimos una función genérica que nos permite pedir un datacube (bbox + fecha + bandas) y descargarlo como GeoTIFF.

In [31]:
# Bandas necesarias confirmadas (según script CyanoLakes para cianobacteria)
BANDAS_TOTAL = ["B02", "B03", "B04", "B05", "B07", "B08", "B8A", "B11", "B12"]
# Índice (1-based, como rasterio) de cada banda dentro del GeoTIFF descargado
IDX = {nombre: i + 1 for i, nombre in enumerate(BANDAS_TOTAL)}
print("Mapa de índices de bandas:", IDX)

# Bandas específicas por índice
BANDS_NDVI = ["B04", "B08"]
BANDS_NDWI = ["B03", "B08"]
BANDS_CYANO = ["B02", "B03", "B04", "B05", "B07", "B08", "B8A", "B11", "B12"]  # CyanoLakes Chlorophyll-a

def descargar_bandas(connection, bbox, fecha, bandas, out_path, coleccion="SENTINEL2_L2A"):
    """Descarga solo las bandas necesarias para un bbox y una fecha puntual."""
    spatial_extent = {
        "west": bbox["west"], "east": bbox["east"],
        "south": bbox["south"], "north": bbox["north"],
    }
    cube = connection.load_collection(
        coleccion,
        spatial_extent=spatial_extent,
        temporal_extent=[fecha, fecha],
        bands=bandas,
    )
    cube = cube.max_time()  # colapsa a una sola fecha/composite
    cube.download(out_path, format="GTiff")
    return out_path

Mapa de índices de bandas: {'B02': 1, 'B03': 2, 'B04': 3, 'B05': 4, 'B07': 5, 'B08': 6, 'B8A': 7, 'B11': 8, 'B12': 9}


In [32]:
# Descargamos todo: para cada fecha de cada lago descargamos todas las bandas en un solo archivo
# (esto va a tomar un rato, hay una petición por cada fecha/lago)

resultados_paths = {}

for nombre_lago, info in lagos.items():
    for fecha in info["fechas"]:
        out_dir = DATA_DIR / nombre_lago
        out_dir.mkdir(exist_ok=True, parents=True)

        # Un solo GeoTIFF con todas las bandas necesarias (NDVI, NDWI, cianobacteria)
        out_path = out_dir / f"{nombre_lago}_{fecha}_bandas.tif"
        descargar_bandas(connection, info, fecha, BANDAS_TOTAL, out_path)

        resultados_paths[(nombre_lago, fecha)] = out_path

print("Total de descargas planeadas:", len(resultados_paths))

KeyboardInterrupt: 

---
## Ejercicio 3 — Cálculo de índices: NDVI, NDWI y cianobacteria

Las fórmulas que vamos a usar:
- **NDVI** = (B08 − B04) / (B08 + B04)
- **NDWI** = (B03 − B08) / (B03 + B08)
- **Cianobacteria:** script CyanoLakes Chlorophyll-a (Kravitz & Matthews, 2020), que usa bandas B02, B03, B04, B05, B07, B08, B8A, B11, B12

Cargamos todas las bandas del GeoTIFF descargado usando los índices definidos en `IDX`.

In [ ]:
def leer_banda(path, indice_banda):
    """Lee una banda específica de un GeoTIFF (índice 1-based como rasterio)."""
    with rasterio.open(path) as src:
        return src.read(indice_banda).astype("float32"), src.profile

def calcular_ndvi(path_bandas):
    """NDVI = (B08 − B04) / (B08 + B04)"""
    b04, _ = leer_banda(path_bandas, IDX["B04"])
    b08, profile = leer_banda(path_bandas, IDX["B08"])
    ndvi = (b08 - b04) / (b08 + b04 + 1e-9)
    return ndvi, profile

def calcular_ndwi(path_bandas):
    """NDWI = (B03 − B08) / (B03 + B08)"""
    b03, _ = leer_banda(path_bandas, IDX["B03"])
    b08, profile = leer_banda(path_bandas, IDX["B08"])
    ndwi = (b03 - b08) / (b03 + b08 + 1e-9)
    return ndwi, profile

def calcular_cyano(path_bandas):
    """
    Cianobacteria: CyanoLakes Chlorophyll-a (Kravitz & Matthews, 2020).
    Usa bandas B02, B03, B04, B05, B07, B08, B8A, B11, B12.
    Aproximación: combinación ponderada de bandas de agua y vegetación.
    """
    # Ejemplo simplificado: chlorophyll-a ~ (B05 - B04) / (B05 + B04)
    # En producción, reemplazar con la fórmula exacta de CyanoLakes
    b04, profile = leer_banda(path_bandas, IDX["B04"])
    b05, _ = leer_banda(path_bandas, IDX["B05"])
    cyano = (b05 - b04) / (b05 + b04 + 1e-9)
    return cyano, profile

In [ ]:
# Aplicamos el cálculo de índices a todas las fechas/lagos

indices_por_fecha = {}

for (nombre_lago, fecha), path_bandas in resultados_paths.items():
    # Verificar si el archivo existe antes de intentar procesarlo
    if not path_bandas.exists():
        print(f"Saltando {nombre_lago} - {fecha}: archivo no descargado aún")
        continue
    
    try:
        ndvi, _ = calcular_ndvi(path_bandas)
        ndwi, _ = calcular_ndwi(path_bandas)
        cyano, _ = calcular_cyano(path_bandas)
        indices_por_fecha[(nombre_lago, fecha)] = {
            "ndvi": ndvi,
            "ndwi": ndwi,
            "cyano": cyano,
            "path": path_bandas
        }
    except Exception as e:
        print(f"Error procesando {nombre_lago} - {fecha}: {e}")
        continue

if len(indices_por_fecha) > 0:
    print(f"Índices calculados para {len(indices_por_fecha)} combinaciones lago/fecha")
else:
    print("No hay archivos descargados aún. Ejecuta la celda de descarga (Ejercicio 2) primero.")

Índices calculados para 2 combinaciones lago/fecha
